# 03 Re-evaluation with unsloth loading: retraining noise

Notebook 02 showed that evaluation is deterministic and that the seed-42 mismatches come from the
adapter loading path. This notebook evaluates all five item-level adapters with the loading path of
the original runs (unsloth `FastLanguageModel.from_pretrained`), so that the remaining differences
of seeds 43-46 measure retraining alone.

| Section | Content | GPU |
|---|---|---|
| [1] | seed 42 on ToMBench, ToMi and OpenToM (Hi-ToM and SocialIQa were run in notebook 02) | yes |
| [2] | seeds 43-46, all five benchmarks | yes |
| [3] | base-model check (notebook 01 `plain_base_*` vs the released baseline) | no |
| [4] | item-level and aggregate noise, comparison over the 25 seed x benchmark cells | no |
| [5] | SocialIQa seed-45 cross-tabulation and margins | no |

Outputs: `repro_unslothload_seed<N>_<benchmark>.csv`, which notebook 06 also uses as the unrotated
predictions. [3]-[5] can be run in a new session without a GPU.

Run the cells in order: [0-1] installs the pinned package versions; restart the runtime
(Runtime > Restart session) and continue from [0-2]. Results are written to Google Drive, and
cells that find their output already saved skip it, so an interrupted run can be resumed by
running [0-2] again and then the interrupted cell.

`DRIVE_DIR` is the Drive folder used by `notebooks/tombench_slm_qlora_complete_pipeline.ipynb`;
`tier4_v2_fixed/` inside it holds that notebook's outputs (split and manifest files, per-item
CSVs, adapters). The other folders are created by the revision notebooks; see
`notebooks/revision/README.md` for where their files are in `results/`.

## [0] Setup

In [ ]:
# -- [0-1] Install packages (versions pinned to requirements.lock.txt) --
import time, torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB VRAM)')
t0 = time.time()
get_ipython().system('pip install -q unsloth==2026.8.18 unsloth_zoo==2026.8.12')
get_ipython().system('pip install -q transformers==5.5.0 trl==0.24.0 accelerate==1.14.0 bitsandbytes==0.50.1 peft==0.20.0')
get_ipython().system('pip install -q "datasets==3.6.0" tqdm')
print(f'   {time.time()-t0:.0f}s  ->  restart the runtime, then run from [0-2]')

In [ ]:
# -- [0-2] Mount Drive, set paths, helpers (evaluate_margin as in notebook 02) --
from google.colab import drive
drive.mount('/content/drive')

import os, re, json as _json, time, hashlib, platform
import pandas as pd, numpy as np, torch
import transformers, peft, unsloth
from unsloth import FastLanguageModel
assert transformers.__version__ == '5.5.0', f'transformers version not pinned: {transformers.__version__}'

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
OUT_DIR_R   = f'{DRIVE_DIR}/repro_check'
os.makedirs(OUT_DIR_R, exist_ok=True)

BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]
MAX_SEQ_LENGTH = 2048

def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()[:16]

ADAPTERS = {s: f'{RESULTS_DIR}/adapter_seed{s}' for s in SEEDS}
ADAPTER_HASH = {}
for s, p in ADAPTERS.items():
    f = f'{p}/adapter_model.safetensors'
    assert os.path.exists(f), f'adapter missing: {f}'
    ADAPTER_HASH[s] = sha16(f)
    print(f'seed{s}: {ADAPTER_HASH[s]}  {p}')
assert ADAPTER_HASH[42] == '1de40b6b5c101d83', 'seed 42 is not the adapter that produced the released CSVs (tier4_v2_fixed) -- stop and check'
# seeds 43-46 are the adapters retrained and saved in notebook 01; record their hashes
with open(f'{OUT_DIR_R}/adapter_hashes_1014.json', 'w') as f:
    _json.dump({f'seed{s}': h for s, h in ADAPTER_HASH.items()}, f, indent=2)

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)
ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')

def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def load_manifest(bm):
    path = f'{RESULTS_DIR}/split_test_full_v2.json' if bm == 'ToMBench' else f'{RESULTS_DIR}/manifest_{bm}.json'
    with open(path, encoding='utf-8') as f:
        return _json.load(f)

MANIFESTS = {bm: load_manifest(bm) for bm in BENCHMARKS}
print({bm: len(v) for bm, v in MANIFESTS.items()})

@torch.no_grad()
def evaluate_margin(eval_records, model, tokenizer, desc):
    # same input and greedy decoding as evaluate() (one item at a time, no padding) + A-E logit gap at the answer step (margin)
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    letter_ids = {}
    for L in 'ABCDE':
        ids = tokenizer.encode(L, add_special_tokens=False)
        assert len(ids) == 1, f'{L} is not a single token: {ids}'
        letter_ids[ids[0]] = L
    lid = torch.tensor(list(letter_ids.keys()))
    rows = []
    for r in tqdm(eval_records, desc=desc):
        inputs = tokenizer(to_text(tokenizer, r['user_prompt']), return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id,
                             output_scores=True, return_dict_in_generate=True)
        gen_ids = out.sequences[0][inputs['input_ids'].shape[1]:]
        gen = tokenizer.decode(gen_ids, skip_special_tokens=True)
        pred = extract_letter(gen)
        margin, top2 = None, None
        for step, tok in enumerate(gen_ids.tolist()):
            if tok in letter_ids and step < len(out.scores):
                sc = out.scores[step][0].float().cpu()[lid]
                v, i = torch.sort(sc, descending=True)
                margin = float(v[0] - v[1])
                top2 = letter_ids[lid[i[1]].item()]
                break
        rows.append({'item_id': r['item_id'], 'answer': r['answer'], 'pred': pred, 'raw': gen,
                     'correct': int(pred == r['answer']), 'letter_margin': margin, 'runner_up': top2})
    return pd.DataFrame(rows)

def out_path(s, bm):
    return f'{OUT_DIR_R}/repro_unslothload_seed{s}_{bm}.csv'

def eval_seed_unsloth(s, benchmarks=BENCHMARKS):
    todo = [bm for bm in benchmarks if not os.path.exists(out_path(s, bm))]
    if not todo:
        print(f'[skip] seed{s}: all files present'); return
    # same loading path as the original runs: pass the adapter folder to FastLanguageModel
    model, tok = FastLanguageModel.from_pretrained(model_name=ADAPTERS[s], max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    for bm in todo:
        t0 = time.time()
        df = evaluate_margin(MANIFESTS[bm], model, tok, desc=f'unsloth-seed{s}-{bm}')
        df.to_csv(out_path(s, bm), index=False)
        print(f'[seed{s}] {bm}: acc={df["correct"].mean()*100:.2f}%  ({(time.time()-t0)/60:.1f} min)')
    del model, tok
    torch.cuda.empty_cache()

meta = {'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda, 'torch': torch.__version__,
        'transformers': transformers.__version__, 'peft': peft.__version__, 'unsloth': unsloth.__version__,
        'python': platform.python_version(), 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
with open(f'{OUT_DIR_R}/session_meta_unsloth_reeval.json', 'w') as f:
    _json.dump(meta, f, indent=2)
print(meta)

## [1] Seed 42, remaining benchmarks

In [ ]:
# -- [1] seed42 --
eval_seed_unsloth(42)
_m = 0; _n = 0
for bm in BENCHMARKS:
    new = pd.read_csv(out_path(42, bm))
    old = pd.read_csv(f'{RESULTS_DIR}/finetuned_seed42_{bm}.csv')
    mg = new.merge(old, on='item_id', suffixes=('_new', '_old'))
    k = int((mg['pred_new'] != mg['pred_old']).sum()); _m += k; _n += len(mg)
    print(f'{bm}: mismatches {k}/{len(mg)}')
print(f'\nseed42, unsloth loading vs released CSVs, total: {_m}/{_n}')
if _m:
    print('Non-zero -- check this before running [2].')

## [2] Seeds 43-46 (replication adapters from notebook 01)

In [ ]:
# -- [2] seed43~46 --
for s in [43, 44, 45, 46]:
    eval_seed_unsloth(s)
print('seeds 43-46 done')

## [3]-[5] Analysis (no GPU needed, reads the CSVs only)

In [ ]:
# -- [3] Paths for the analysis and a check of the base model --
from google.colab import drive
drive.mount('/content/drive')
import os, json as _json, pandas as pd, numpy as np
DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
OUT_DIR_P2  = f'{DRIVE_DIR}/cyclic_shift_check'
OUT_DIR_R   = f'{DRIVE_DIR}/repro_check'
BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]
pd.set_option('display.width', 200)

def mismatch(a, b):
    mg = pd.read_csv(a).merge(pd.read_csv(b), on='item_id', suffixes=('_a', '_b'))
    return mg, int((mg['pred_a'] != mg['pred_b']).sum())

# the base model has no adapter, so its re-evaluation in notebook 01 should equal the released baseline
print('[base] notebook 01 plain_base vs released baseline')
for bm in BENCHMARKS:
    p1, p2 = f'{OUT_DIR_P2}/plain_base_{bm}.csv', f'{RESULTS_DIR}/baseline_{bm}.csv'
    if os.path.exists(p1) and os.path.exists(p2):
        mg, k = mismatch(p1, p2)
        print(f'  {bm}: mismatches {k}/{len(mg)}')
    else:
        print(f'  {bm}: file missing')

In [ ]:
# -- [4] Item-level and aggregate noise --
rows = []
for s in SEEDS:
    for bm in BENCHMARKS:
        p_new = f'{OUT_DIR_R}/repro_unslothload_seed{s}_{bm}.csv'
        p_pub = f'{RESULTS_DIR}/finetuned_seed{s}_{bm}.csv'
        p_peft = f'{OUT_DIR_P2}/plain_seed{s}_{bm}.csv'      # PeftModel loading (notebook 01)
        p_base = f'{RESULTS_DIR}/baseline_{bm}.csv'
        if not (os.path.exists(p_new) and os.path.exists(p_pub)):
            print(f'[missing] seed{s} {bm}'); continue
        mg, k = mismatch(p_new, p_pub)
        k_peft = mismatch(p_peft, p_pub)[1] if os.path.exists(p_peft) else None
        base_acc = pd.read_csv(p_base)['correct'].mean() * 100
        new_acc, pub_acc = mg['correct_a'].mean() * 100, mg['correct_b'].mean() * 100
        rows.append({'seed': s, 'benchmark': bm, 'n': len(mg),
                     'mismatch_unsloth': k, 'mismatch_peft_1011': k_peft,
                     'new_delta_pp': new_acc - base_acc, 'orig_delta_pp': pub_acc - base_acc,
                     'delta_diff_pp': new_acc - pub_acc})
res = pd.DataFrame(rows)
res.to_csv(f'{OUT_DIR_R}/unsloth_reeval_by_cell.csv', index=False)

print('[by cell] unsloth loading vs released CSVs (mismatches with PeftModel loading alongside)')
print(res.round(2).to_string(index=False))

print('\n[item level] mismatch rate by seed (%)')
it = res.groupby('seed')[['mismatch_unsloth', 'mismatch_peft_1011', 'n']].sum(min_count=1)
it['rate_unsloth_pct'] = (it['mismatch_unsloth'] / it['n'] * 100).round(2)
it['rate_peft_1011_pct'] = (it['mismatch_peft_1011'] / it['n'] * 100).round(2)
print(it)
r = res[res['seed'] != 42]
if len(r):
    tot = r[['mismatch_unsloth', 'n']].sum()
    print(f'seeds 43-46 (retraining only): {tot["mismatch_unsloth"] / tot["n"] * 100:.2f}%')
it.to_csv(f'{OUT_DIR_R}/unsloth_reeval_item_level.csv')

print('\n[aggregate] difference in change (pp) = re-evaluated accuracy - released accuracy (same released baseline)')
agg = []
for name, g in [('seed42 (no retraining)', res[res['seed'] == 42]), ('seed43-46 (retrained)', r)]:
    d = g['delta_diff_pp']
    if len(d):
        agg.append({'condition': name, 'n_cells': len(d), 'mean_signed_diff_pp': d.mean(), 'mean_abs_diff_pp': d.abs().mean(),
                    'SD': d.std(ddof=1) if len(d) > 1 else np.nan,
                    'p90_abs_diff_pp': d.abs().quantile(0.9), 'max_abs_diff_pp': d.abs().max(),
                    'sign_changed_cells': int(((g['new_delta_pp'] > 0) != (g['orig_delta_pp'] > 0)).sum())})
agg = pd.DataFrame(agg)
print(agg.round(2).to_string(index=False))
agg.to_csv(f'{OUT_DIR_R}/unsloth_reeval_aggregate.csv', index=False)

In [ ]:
# -- [5] SocialIQa seed 45 cross-tabulation and margins --
p_new = f'{OUT_DIR_R}/repro_unslothload_seed45_SocialIQa.csv'
mg, k = mismatch(p_new, f'{RESULTS_DIR}/finetuned_seed45_SocialIQa.csv')
d = mg[mg['pred_a'] != mg['pred_b']]
print(f'SocialIQa seed45 (unsloth loading): mismatches {k}/{len(mg)}')
ct = pd.crosstab(d['pred_b'], d['pred_a'], margins=True)
ct.index.name, ct.columns.name = 'released_pred', 'reeval_unsloth'
print(ct)
ct.to_csv(f'{OUT_DIR_R}/socialiqa_seed45_flip_crosstab_unsloth.csv')
lost = int(((d['correct_b'] == 1) & (d['correct_a'] == 0)).sum())
gain = int(((d['correct_b'] == 0) & (d['correct_a'] == 1)).sum())
print(f'correct->wrong {lost}, wrong->correct {gain}, net {gain - lost} items = {(gain - lost) / len(mg) * 100:+.2f}pp')

rows = []
for s in SEEDS:
    p = f'{OUT_DIR_R}/repro_unslothload_seed{s}_SocialIQa.csv'
    if not os.path.exists(p): continue
    m2, _ = mismatch(p, f'{RESULTS_DIR}/finetuned_seed{s}_SocialIQa.csv')
    dd = m2[m2['pred_a'] != m2['pred_b']]
    pair = (dd['pred_b'] + '->' + dd['pred_a']).value_counts()
    rows.append({'seed': s, 'n_flip': len(dd),
                 'to_A_pct': (dd['pred_a'] == 'A').mean() * 100 if len(dd) else np.nan,
                 'top_direction': pair.index[0] if len(pair) else None,
                 'top_share_pct': pair.iloc[0] / len(dd) * 100 if len(dd) else np.nan,
                 'net_correct': int(dd['correct_a'].sum() - dd['correct_b'].sum())})
sd = pd.DataFrame(rows)
print('\n[SocialIQa flip direction by seed (unsloth loading)]')
print(sd.round(1).to_string(index=False))
sd.to_csv(f'{OUT_DIR_R}/socialiqa_flip_direction_by_seed_unsloth.csv', index=False)

# margin: are the items that changed with the retrained adapters near ties?
mm = []
for s in [43, 44, 45, 46]:
    for bm in BENCHMARKS:
        p = f'{OUT_DIR_R}/repro_unslothload_seed{s}_{bm}.csv'
        if not os.path.exists(p): continue
        m3 = pd.read_csv(p).merge(pd.read_csv(f'{RESULTS_DIR}/finetuned_seed{s}_{bm}.csv')[['item_id', 'pred']]
                                  .rename(columns={'pred': 'pred_pub'}), on='item_id')
        m3['flipped'] = m3['pred'] != m3['pred_pub']; m3['seed'] = s; m3['benchmark'] = bm
        mm.append(m3)
if mm:
    mm = pd.concat(mm)
    print('\n[letter margin, seeds 43-46] items that differ from the released CSVs vs items that do not')
    print(mm.groupby('flipped')['letter_margin'].describe()[['count', 'mean', '50%', 'max']].round(3))
    f = mm[mm['flipped']]
    for th in [0.5, 1.0, 2.0]:
        print(f'changed items with margin < {th}: {(f["letter_margin"] < th).mean()*100:.1f}%')
    print(f'changed items whose released prediction is the runner-up letter: {(f["runner_up"] == f["pred_pub"]).mean()*100:.1f}%')
    mm.to_csv(f'{OUT_DIR_R}/unsloth_reeval_margin_seed43_46.csv', index=False)